# IP 02 — Integrated Prediction Template

Run this after the future inference-feature builder has prepared a model-compatible feature dataset.

In [1]:
# Import libraries
from pathlib import Path
import sys
import json
import yaml
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while (
    not (PROJECT_ROOT / "src").exists()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = (PROJECT_ROOT / "configs" / "integrated_pipeline.yaml")

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/integrated_pipeline.yaml')

In [3]:
# Import module for running Integrated Pipeline
from src.ontario_peak_risk.modeling.common import (
    load_feature_dataset,
    load_modeling_config,
)

from src.ontario_peak_risk.inference.integrated_pipeline import (
    IntegratedPredictionPipeline,
    save_prediction_outputs,
)


In [4]:
FOUNDATION_PATH = (
    PROJECT_ROOT
    / "configs"
    / "modeling_foundation.yaml"
)

modeling_config, _ = load_modeling_config(
    FOUNDATION_PATH
)

feature_dataset = load_feature_dataset(
    modeling_config,
    PROJECT_ROOT,
)

feature_dataset["timestamp"] = pd.to_datetime(
    feature_dataset["timestamp"]
)

print("Feature dataset:", feature_dataset.shape)
print(
    "Available period:",
    feature_dataset["timestamp"].min(),
    "to",
    feature_dataset["timestamp"].max(),
)
print(
    "FSAs:",
    sorted(feature_dataset["fsa"].dropna().unique().tolist()),
)

Feature dataset: (262944, 100)
Available period: 2021-01-01 00:00:00 to 2025-12-31 23:00:00
FSAs: ['L4T', 'M5R', 'M5S', 'M6G', 'M9R', 'M9W']


In [5]:
# ------------------------------------------------------------
# Historical replay forecast origin
# ------------------------------------------------------------
# This is a pipeline smoke test, NOT another model evaluation.
# We choose the latest timestamp that still has a complete
# 24-hour feature window in the historical feature dataset.

available_fsas = sorted(
    feature_dataset["fsa"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

max_timestamp = feature_dataset["timestamp"].max()

forecast_origin = (
    max_timestamp
    - pd.Timedelta(hours=24)
)

# Prefer an exact timestamp present for all FSAs.
candidate_origins = (
    feature_dataset.loc[
        feature_dataset["timestamp"] <= forecast_origin,
        ["fsa", "timestamp"],
    ]
    .groupby("timestamp", observed=True)["fsa"]
    .nunique()
)

candidate_origins = candidate_origins[
    candidate_origins >= len(available_fsas)
]

if candidate_origins.empty:
    raise ValueError(
        "Could not find a historical replay origin with all FSAs available."
    )

forecast_origin = candidate_origins.index.max()

print("Historical replay forecast origin:", forecast_origin)
print("FSAs:", available_fsas)

Historical replay forecast origin: 2025-12-30 23:00:00
FSAs: ['L4T', 'M5R', 'M5S', 'M6G', 'M9R', 'M9W']


In [6]:
# Defin artifacts
ARTIFACTS = (
    PROJECT_ROOT
    / CONFIG["paths"]["artifacts_dir"]
)

#Load pipeline
pipeline = IntegratedPredictionPipeline(
    ARTIFACTS
)


# Example only:
# inference_feature_dataset = ...
# output = pipeline.predict(
#     forecast_origin="2026-01-15 00:00:00",
#     fsas=["L4T", "M5R", "M5S", "M6G", "M9R", "M9W"],
#     inference_feature_dataset=inference_feature_dataset,
# )
# display(output.head(30))

output = pipeline.predict(
    forecast_origin=forecast_origin,
    fsas=available_fsas,
    inference_feature_dataset=feature_dataset,
)

display(output.head(30))





,fsa,forecast_origin,target_timestamp,horizon,forecast_consumption_kwh,peak_risk_score,peak_alert
0,L4T,2025-12-30 23:00:00,2025-12-31 00:00:00,1,11391.804060,0.000132,False
1,L4T,2025-12-30 23:00:00,2025-12-31 01:00:00,2,10575.189247,0.000073,False
2,L4T,2025-12-30 23:00:00,2025-12-31 02:00:00,3,9967.082221,0.000057,False
3,L4T,2025-12-30 23:00:00,2025-12-31 03:00:00,4,9662.285006,0.000113,False
4,L4T,2025-12-30 23:00:00,2025-12-31 04:00:00,5,9579.000471,0.000128,False
5,L4T,2025-12-30 23:00:00,2025-12-31 05:00:00,6,9780.371439,0.000157,False
6,L4T,2025-12-30 23:00:00,2025-12-31 06:00:00,7,9966.185314,0.000100,False
7,L4T,2025-12-30 23:00:00,2025-12-31 07:00:00,8,10458.347517,0.000066,False
8,L4T,2025-12-30 23:00:00,2025-12-31 08:00:00,9,11145.486643,0.000220,False
9,L4T,2025-12-30 23:00:00,2025-12-31 09:00:00,10,11855.571180,0.000246,False


In [7]:
# Display pipeline
pipeline

In [8]:
print("Rows:", len(output))
print("Expected rows:", len(available_fsas) * 24)
print("Horizons:", output["horizon"].min(), "to", output["horizon"].max())
print("FSA count:", output["fsa"].nunique())
print("Peak alerts:", int(output["peak_alert"].sum()))

Rows: 144
Expected rows: 144
Horizons: 1 to 24
FSA count: 6
Peak alerts: 50


In [9]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / CONFIG["paths"]["outputs_dir"]
    / "historical_replay_demo"
)

save_prediction_outputs(
    output,
    OUTPUT_DIR,
)

summary = (
    output
    .groupby("fsa", observed=True)
    .agg(
        max_forecast_consumption_kwh=(
            "forecast_consumption_kwh",
            "max",
        ),
        max_peak_risk_score=(
            "peak_risk_score",
            "max",
        ),
        peak_alert_hours=(
            "peak_alert",
            "sum",
        ),
    )
    .reset_index()
)

summary.to_csv(
    OUTPUT_DIR / "integrated_24h_summary_by_fsa.csv",
    index=False,
)

display(summary)

print("Saved outputs to:", OUTPUT_DIR)

,fsa,max_forecast_consumption_kwh,max_peak_risk_score,peak_alert_hours
0,L4T,14865.533818,0.415957,6
1,M5R,13135.196049,0.988430,12
2,M5S,6963.333033,0.991410,11
3,M6G,17741.693093,0.951889,6
4,M9R,8212.401477,0.960627,7
5,M9W,18804.563871,0.971639,8


Saved outputs to: e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\outputs\integrated_pipeline\historical_replay_demo


### Interpretation

This is a **historical replay / smoke test** to demonstrate that the 48 serialized model artifacts can be loaded and executed through a single logical pipeline. It is not a new holdout evaluation and should not be used to change model parameters.
